In [ ]:
# # ── OPTIONAL RESET — run once, then re-comment ──────────────────────────────
# # Clears the %store-persisted ids (kb_id, gateway_id) and any stale in-memory
# # handles from a previous run, so the Setup cells start from a clean slate and
# # create everything fresh. Use after you've manually deleted resources (KB,
# # Gateway, Runtime) out-of-band. This does NOT delete AWS resources — it only
# # forgets local references to them. For actual teardown, use the Cleanup cell.
# for _k in ('kb_id', 'gateway_id'):
#     try: get_ipython().run_line_magic('store', f'-d {_k}')
#     except Exception: pass
# for _v in ('kb', 'kb_id', 'gateway_id', 'gateway_url', 'agent_arn', 'agent_id',
#            'agentcore_runtime', 'launch_result', 'exec_role_name', 'spans',
#            'session_span_logs'):
#     globals().pop(_v, None)
# print('Reset done — Setup cells will create fresh KB + Gateway + Runtime.')

# Evaluating a Managed Knowledge Base — Agentic RAG on AgentCore

**How do you measure the quality, cost, and behavior of a Bedrock *Managed* Knowledge Base
serving agentic RAG?** This notebook completes the observability story: it deploys a Strands
agent (KB behind an AgentCore Gateway) to **AgentCore Runtime**, drives evaluation queries
through it, and then reads **all seven telemetry layers** — from raw KB metrics up to
LLM-as-a-judge quality scores.

It is the companion to the [observability notebook](../../05-Observability/02-agentcore-observability.ipynb),
which covered Layers 1–5 with a *local* agent. Here we **deploy** the agent, which unlocks the
two layers a local agent can't produce:

- **Layer 6 — cost & quota:** the deployed agent's OTEL spans carry per-call `gen_ai.usage`
  tokens, so cost is *measured*, not estimated.
- **Layer 7 — quality:** AgentCore **Evaluate** scores the session's span tree
  (Faithfulness, Correctness, Tool-Selection Accuracy, …). 

## The 7-layer evaluation taxonomy

Every signal an agentic Managed-KB application produces. Layers 1–5 are *operational
telemetry* (covered in depth in notebook 02); **this notebook adds Layers 6–7** and shows all
seven end-to-end on a deployed agent.

| # | Layer | What it captures | Source | Here? |
|---|-------|------------------|--------|-------|
| 1 | **KB-native metrics** | Invocations, errors, throttles, TotalIterationCount, RawDataSize | `AWS/Bedrock/KnowledgeBases` | ✅ |
| 2 | **KB ingestion** | Docs scanned / indexed / failed; per-doc crawl→sync→index | ingestion-job API + `APPLICATION_LOGS` | ✅ |
| 3 | **Agentic retrieval quality** | Retrieval-set (chunks, coverage, duplication) + grounding (citation coverage, utilization) | agent's `AgenticRetrieveStream` payload in the runtime log | ✅ |
| 4 | **Gateway / MCP** | Invocations, Latency, errors, overhead | `AWS/Bedrock-AgentCore` | ✅ |
| 5 | **Traces (spans)** | Agent span tree, per-op latency, Gateway overhead | `aws/spans` | ✅ |
| 6 | **Cost & quota** ⭐ | Per-call tokens → **$ cost (flat)** + **quota consumed (burndown)** | deployed-agent OTEL spans (`gen_ai.usage.*`) | ✅ |
| 7 | **Quality** ⭐ | Faithfulness, Correctness, Tool-Selection Accuracy, … | AgentCore **Evaluate** (LLM-as-judge) | ✅ |

> **Cost ≠ quota (a common trap):** you're billed on *actual* tokens (`input×price + output×price`).
> The **burndown** multiplier (Claude v4.8=15×, Sonnet 5=10×, v4.7-and-below=5×, others=1×) applies
> only to **TPM/TPD quota** — *not* the bill. We report both, separately.

## Architecture — where each layer originates

```
                        ┌─── Layer 1: AWS/Bedrock/KnowledgeBases (metrics)
   S3 ──▶ Managed KB ───┼─── Layer 2: ingestion job + APPLICATION_LOGS
                        └─── Layer 3: AgenticRetrieveStream payload (chunks + cited answer)
        ▲
        │ MCP (AgenticRetrieveStream)
   AgentCore Gateway ───── Layer 4: AWS/Bedrock-AgentCore (metrics)
        ▲
        │ MCP (SigV4)
   Strands agent  ┌─────── Layer 5: OTEL span tree ──▶ aws/spans
   on AgentCore ──┼─────── Layer 6: gen_ai.usage tokens on spans → cost + quota
   RUNTIME        └─────── Layer 7: AgentCore Evaluate scores the session spans
        ▲
   invoke_agent_runtime(session_id)   ← session.id is the join key across every layer
```

The agent is **deployed** (not local): the Runtime host auto-instruments it with OTEL, so its
span tree carries token usage (Layer 6) and is scorable by Evaluate (Layer 7). The Gateway
exposes the KB as an **`AgenticRetrieveStream`** tool — the agent asks a question and Bedrock
plans, retrieves, reranks, and returns a synthesized, cited answer (Layer 3 reads that payload
back out of the runtime log).

## Prerequisites

- AWS creds with Bedrock, IAM, AgentCore (Runtime + Gateway), CloudWatch, X-Ray, ECR, CodeBuild
- Model access for Claude Haiku; **CloudWatch Transaction Search** (enabled below)
- Python 3.10+. Deploying builds a container via CodeBuild (ARM64, no local Docker needed). 

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet
%pip install strands-agents strands-agents-tools mcp-proxy-for-aws bedrock-agentcore bedrock-agentcore-starter-toolkit --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

## Setup — configuration

We reuse the shared helpers from the observability notebook (`kb_observability`, `nb_display`)
plus the cost/quota helper (`pricing`). Names carry a run suffix; a stored KB/Gateway is reused
if it still exists. 

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, boto3, sys, time, uuid, json
import pandas as pd

# Credentials come from the ambient AWS chain. Point at your own account/region by
# exporting them in the shell BEFORE launching Jupyter, e.g.:

os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')

sys.path.insert(0, '../..')
from utils.managed_knowledge_base import ManagedKnowledgeBase
from utils import kb_observability as obs
from utils import nb_display as ui
from utils.pricing import compute_cost_usd, quota_consumed

region = boto3.session.Session().region_name or os.environ['AWS_DEFAULT_REGION']
account_id = boto3.session.Session().client('sts').get_caller_identity()['Account']
suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]
region_prefix = next((v for k, v in {'us-':'us','eu-':'eu','ap-':'apac'}.items() if region.startswith(k)), 'us')
model_id = f'{region_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

# runtimeSessionId must be >= 33 chars — the join key across every layer.
session_id = f'bmkb-eval-{uuid.uuid4().hex}{uuid.uuid4().hex[:8]}'
kb_name = f'bmkb-eval-{suffix}'
bucket_name = f'bedrock-bmkb-eval-{suffix}-{account_id}'
gateway_name = f'bmkb-eval-gw-{suffix}'
agent_name = f'bmkb_eval_agent_{suffix}'

ui.rule('Configuration')
ui.ok(f'Region: {region}   Account: {account_id}')
ui.ok(f'Model: {model_id}')
ui.ok(f'Session ID: {session_id}  (len={len(session_id)})')

## Setup — Managed Knowledge Base

Reuse a stored KB if it still exists (validated via `get_knowledge_base`), else create one and
ingest the Octank sample. This also gives us **Layer 2** (ingestion) shortly. 

In [ ]:
s3 = boto3.client('s3')
try:
    kb_id
except NameError:
    try:
        %store -r kb_id
    except Exception:
        pass
kb_id = kb_id if 'kb_id' in dir() else None

if kb_id:
    try:
        boto3.client('bedrock-agent', region_name=region).get_knowledge_base(knowledgeBaseId=kb_id)
        ui.ok(f'Reusing stored KB: {kb_id}')
        kb = ManagedKnowledgeBase.from_existing(kb_id, region_name=region)
    except Exception:
        ui.info(f'Stored KB {kb_id} gone — creating fresh.'); kb_id = None

if not kb_id:
    if region == 'us-east-1':
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})
    s3.upload_file('../../synthetic_dataset/octank_financial_10K.pdf', bucket_name, 'octank_financial_10K.pdf')
    kb = ManagedKnowledgeBase(kb_name=kb_name, bucket_name=bucket_name,
                              enable_logging=True, region_name=region, suffix=suffix)
    kb_id = kb.kb_id
    %store kb_id
    time.sleep(30)
    kb.start_ingestion_job()
ui.ok(f'KB ID: {kb_id}')

## Setup — AgentCore Gateway + telemetry delivery

The Gateway exposes the KB as an MCP tool the deployed agent calls. We also enable trace
delivery (90% sampling) so the agent's spans land in `aws/spans` for Layers 5–7. 

In [ ]:
ac = kb.get_agentcore_client()
try:
    %store -r gateway_id
except Exception:
    pass
gateway_id = gateway_id if 'gateway_id' in dir() else None
gateway_url = None
if gateway_id:
    try:
        g = ac.get_gateway(gatewayIdentifier=gateway_id); gateway_url = g['gatewayUrl']
        ui.ok(f'Reusing Gateway: {gateway_id} ({g["status"]})')
    except Exception:
        gateway_id = None

gw_role_name = f'AmazonBedrockGatewayRole_{suffix}'
iam = boto3.client('iam')
if not gateway_id:
    try:
        role = iam.create_role(RoleName=gw_role_name,
            AssumeRolePolicyDocument=json.dumps({'Version':'2012-10-17','Statement':[{
                'Effect':'Allow','Principal':{'Service':'bedrock-agentcore.amazonaws.com'},
                'Action':'sts:AssumeRole','Condition':{'StringEquals':{'aws:SourceAccount':account_id}}}]}))
    except iam.exceptions.EntityAlreadyExistsException:
        role = iam.get_role(RoleName=gw_role_name)
    # The Gateway assumes this role to call the KB. IMPORTANT: bedrock:AgenticRetrieveStream
    # is NOT resource-scopable — it must be granted on "*" (per the connector docs). The
    # single-search actions stay scoped to the KB resource.
    iam.put_role_policy(RoleName=gw_role_name, PolicyName='kb-retrieve',
        PolicyDocument=json.dumps({'Version':'2012-10-17','Statement':[
            {'Effect':'Allow',
             'Action':['bedrock:Retrieve','bedrock:RetrieveAndGenerate','bedrock:GetKnowledgeBase',
                       'bedrock:ListKnowledgeBases'],
             'Resource':f'arn:aws:bedrock:{region}:{account_id}:knowledge-base/*'},
            {'Effect':'Allow',
             'Action':['bedrock:AgenticRetrieveStream'],
             'Resource':'*'},
        ]}))
    time.sleep(20)
    gw = kb.create_gateway(gateway_name=gateway_name, gateway_role_arn=role['Role']['Arn'], auth_type='AWS_IAM')
    gateway_id, gateway_url = gw['gateway_id'], gw['gateway_url']
    # Expose the AGENTIC tool (multi-step planning + rerank + a synthesized, cited answer),
    # not plain Retrieve. foundationModelType=MANAGED → the service orchestrates, so there is
    # no model ARN / inference profile for us to manage here.
    kb.create_gateway_kb_target(gateway_id=gateway_id, target_name='kb-retrieve',
                                tool='AgenticRetrieveStream')
    %store gateway_id
ui.ok(f'Gateway: {gateway_id}')

logs = boto3.client('logs', region_name=region); xray = boto3.client('xray', region_name=region)
def _ok(fn, label):
    try: fn(); ui.ok(label)
    except Exception as e:
        ui.ok(f'{label} (already set)') if any(k in str(e).lower() for k in ['already','conflict']) else ui.info(f'{label}: {e}')
_ok(lambda: xray.update_trace_segment_destination(Destination='CloudWatchLogs'), 'Traces → CloudWatch Logs')
_ok(lambda: xray.update_indexing_rule(Name='Default', Rule={'Probabilistic':{'DesiredSamplingPercentage':90}}), 'Span sampling → 90%')
gw_arn = f'arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/{gateway_id}'
_ok(lambda: logs.put_delivery_source(name=f'gw-{gateway_id}-traces', logType='TRACES', resourceArn=gw_arn), 'Gateway traces source')
gdest = None
try: gdest = logs.put_delivery_destination(name=f'gw-{gateway_id}-xray', deliveryDestinationType='XRAY')
except Exception: pass
if gdest:
    _ok(lambda: logs.create_delivery(deliverySourceName=f'gw-{gateway_id}-traces',
        deliveryDestinationArn=gdest['deliveryDestination']['arn']), 'Gateway traces delivery')

## Setup — deploy the agent to AgentCore Runtime

`%%writefile` writes literal text (no interpolation), so the agent reads Gateway URL + model
from **environment variables** passed at launch. `BedrockAgentCoreApp` + `@app.entrypoint` is
the Runtime contract; the host auto-instruments it with OTEL (that's what powers Layers 5–7). 

In [ ]:
%%writefile agent_eval.py
# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: MIT-0
"""Strands agent on AgentCore Runtime — answers via a Managed KB behind an AgentCore
Gateway (MCP). Config comes from env vars set at launch."""
import os
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.tools.mcp import MCPClient
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

app = BedrockAgentCoreApp()
REGION = os.environ["AWS_REGION"]
GATEWAY_URL = os.environ["GATEWAY_URL"]
MODEL_ID = os.environ["MODEL_ID"]
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

@app.entrypoint
def invoke(payload):
    prompt = payload.get("prompt", "")
    mcp_client = MCPClient(lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_URL, aws_region=REGION, aws_service="bedrock-agentcore"))
    with mcp_client:
        agent = Agent(model=model, tools=mcp_client.list_tools_sync(),
                      system_prompt="Answer using the knowledge base tool. Cite sources.")
        return agent(prompt).message["content"][0]["text"]

if __name__ == "__main__":
    app.run()

In [ ]:
%%writefile requirements_agent.txt
strands-agents
strands-agents-tools
mcp-proxy-for-aws
bedrock-agentcore

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint='agent_eval.py', auto_create_execution_role=True, auto_create_ecr=True,
    requirements_file='requirements_agent.txt', region=region, agent_name=agent_name)
ui.ok(f'Configured runtime: {agent_name}')

launch_result = agentcore_runtime.launch(env_vars={'GATEWAY_URL': gateway_url, 'MODEL_ID': model_id})
agent_arn = launch_result.agent_arn
ui.ok(f'Launched: {agent_arn}')

end_states = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
status = agentcore_runtime.status().endpoint['status']
while status not in end_states:
    time.sleep(15); status = agentcore_runtime.status().endpoint['status']; print('  ', status)
ui.ok(f'Runtime status: {status}')

In [ ]:
# Grant the auto-created execution role permission to invoke the Gateway.
exec_role_arn = agentcore_runtime.status().agent.get('roleArn') or getattr(launch_result, 'execution_role', None)
exec_role_name = exec_role_arn.split('/')[-1]
iam.put_role_policy(RoleName=exec_role_name, PolicyName='invoke-gateway',
    PolicyDocument=json.dumps({'Version':'2012-10-17','Statement':[{'Effect':'Allow',
        'Action':['bedrock-agentcore:InvokeGateway'],
        'Resource':f'arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/{gateway_id}'}]}))
ui.ok(f'Granted InvokeGateway to {exec_role_name}')
time.sleep(20)
# agent_id (used for Layer-7 span download) = the ARN's runtime name.
agent_id = agent_arn.split('/')[-1]
ui.ok(f'Agent id: {agent_id}')

## Generate evaluation traffic

Invoke the deployed agent with our `session_id` (the Runtime stamps it on every span). These
queries populate all seven layers. 

In [ ]:
agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
queries = [
    "What is Octank Financial's total revenue?",
    "What are Octank's main risk factors?",
    "Describe Octank's growth strategy.",
]
for q in queries:
    r = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn, qualifier='DEFAULT', runtimeSessionId=session_id,
        payload=json.dumps({'prompt': q}))
    body = r['response'].read() if hasattr(r['response'], 'read') else b''.join(r['response'])
    try: ans = json.loads(body.decode('utf-8'))
    except Exception: ans = body.decode('utf-8', 'replace')
    ui.ok(f'Q: {q[:45]}  →  {str(ans)[:70]}...')
ui.info('Waiting 120s for metrics + spans to land...')
time.sleep(120)

## Layer 1 — KB-native metrics

> **📊 What to expect** — the KB's own operational counters (Sum over the window):
> `Invocations` **≥ 1** · `ClientErrors`/`ServerErrors` **0** (published only when they occur) ·
> `Throttles` **0** · `RawDataSize` a small GB figure. ⚠️ **No `Latency`** here — KB latency lives
> in the traces (Layer 5).

Bedrock publishes these to `AWS/Bedrock/KnowledgeBases` automatically. A clean run shows mainly
`Invocations` — errors/throttles are published only when they occur, and there is **no `Latency`
metric** here (KB latency lives in traces, Layer 5). 

In [ ]:
ui.rule('Layer 1 — KB-native metrics')
display(ui.style_metrics(obs.get_kb_metrics(kb_id, region_name=region, hours=1)))

## Layer 2 — KB ingestion

> **📊 What to expect** — proof the corpus indexed cleanly. The job stats show
> `numberOfDocumentsScanned` = `numberOfNewDocumentsIndexed` (here **1** = **1**) and
> `numberOfDocumentsFailed` **0**; the per-doc log shows `crawl_status` / `index_status` = **SUCCESS**.
> Any non-zero *failed* here means retrieval later will be missing content.

Two views: the authoritative **ingestion-job statistics** (docs scanned / indexed / failed) and
the per-document **crawl → sync → index** log detail. 

In [ ]:
ui.rule('Layer 2 — KB ingestion')
ui.ok('Ingestion job statistics:')
display(obs.get_ingestion_stats(kb_id, region_name=region))
rows = obs.query_ingestion_logs(kb_id, region_name=region, hours=24)
if rows:
    display(pd.DataFrame([{f['field']: f['value'] for f in row if f['field'] != '@ptr'} for row in rows]))
else:
    ui.info('No per-document logs yet.')

## Layer 3 — Agentic retrieval quality

> **📊 What to expect** — one row per query, all **reference-free** (no ground truth):
> `chunk_count` **10** · `distinct_docs` **1** (single-doc corpus) · `duplicate_rate` **0** ·
> `retrieval_utilization` **~0.2–0.9** (cited / retrieved — a precision proxy; higher for broad
> questions) · `grounded_coverage` **~0.99** (answer backed by citations — near 1.0 = low
> hallucination risk). No per-chunk relevance `score` — see the note at the end for why.

Because the agent's KB tool is **`AgenticRetrieveStream`** (not plain `Retrieve`), the retrieval
result is different — and so are the metrics we can compute from it. This section explains, for
someone seeing it for the first time, **where the data comes from, what we compute, and what each
number means.**

### Where the data comes from (this is the important part)

The agent calls its KB tool *through the Gateway*; we do **not** make a second retrieval call just
to measure. Instead we read what the agent **actually retrieved**, from its own telemetry. But the
data isn't all in one place — here's the map (verified live, `us-west-2`):

| We want… | Is it on the tool-call span? | Where it actually lives |
|---|---|---|
| that a retrieval happened | ✅ `execute_tool …___AgenticRetrieveStream` span | span attribute `gen_ai.tool.name`, `gen_ai.tool.status` |
| the retrieved **chunks** + the **cited answer** | ❌ *not on the span* | **runtime log**, scope `opentelemetry.instrumentation.botocore.bedrock-runtime`, inside `body.content[].text` (a JSON string) |
| the **planning / iteration** trace steps | ❌ nowhere | streamed over MCP `notifications/message`, **never persisted** — so there is no iteration metric here |

The payload records carry no `session.id`, so we tie them to *this run* by **`traceId`**: we take
the traceIds of this session's spans (Layer 5) and keep only payloads whose record shares one.
`obs.fetch_agentic_retrievals()` does exactly this; `obs.extract_agentic_retrieval_quality()` then
computes the metrics below from each payload.

### What we compute, and what it means

**A · Retrieval-set** — *what the retriever pulled* (from `results[]`):

| Metric | From | Meaning |
|---|---|---|
| `chunk_count` | `len(results)` | how many chunks came back |
| `distinct_docs` | unique `metadata._document_id` | source coverage (1 = all from one file) |
| `duplicate_rate` | `1 − distinct_chunks/chunk_count` | redundant retrieval (same chunk twice) |
| `avg_chunk_chars`, `total_context_chars` | `len(content.text)` | how much context was fed to the model (budget signal) |

**B · Grounding / citation** — *how well the answer used what was retrieved* (from `generatedResponse`):

| Metric | From | Meaning |
|---|---|---|
| `num_citations` | `len(citations)` | how many answer spans are cited |
| `cited_chunks` | distinct `citations[].references[].resultIndex` | how many of the retrieved chunks the answer actually used |
| `retrieval_utilization` | `cited_chunks / chunk_count` | **precision proxy** — low = the retriever over-fetched (e.g. `0.2` = only 2 of 10 chunks used) |
| `grounded_coverage` | cited answer-span chars / total answer chars | **reference-free faithfulness** — fraction of the answer backed by a citation (near `1.0` = well-grounded, low hallucination risk) |
| `answer_chars` | `len(answer)` | answer length **in characters** (verbosity — *not* tokens) |

> **Why no relevance scores here?** Plain `Retrieve` returns a per-chunk `score` (0–1); we used those
> in notebook 02. `AgenticRetrieveStream` returns **no score** — it reranks internally and hands back a
> synthesized answer instead. So Layer 3 shifts from *"how relevant is each chunk"* to *"how much of
> what we retrieved did the answer actually use, and how well is it grounded"* — arguably the more
> useful RAG signal. All of it is reference-free: no ground-truth labels, no extra LLM-judge cost.

In [ ]:
ui.rule('Layer 3 — Agentic retrieval quality')
# We read what the DEPLOYED AGENT actually retrieved — no extra retrieval call.
# Step 1: get this session's spans and collect their traceIds (Layer 3 fetches its own
#         spans so it runs before Layer 5; the Layer-5 cell re-queries for the full tree).
_l3_spans = obs.query_spans(region_name=region, hours=1, limit=200)
session_trace_ids = {s.get('traceId') for s in _l3_spans
                     if s.get('attributes', {}).get('session.id') == session_id and s.get('traceId')}
# Step 2: pull the agentic-retrieve payloads from the agent's runtime log, correlated
#         to this session by traceId (see utils.kb_observability.fetch_agentic_retrievals
#         for exactly where in the logs these live).
retrievals = obs.fetch_agentic_retrievals(agent_id, session_trace_ids, region_name=region, hours=1)

if not retrievals:
    ui.info('No agentic-retrieve payloads yet — spans/logs may still be landing; re-run in ~1 min.')
else:
    quality = [obs.extract_agentic_retrieval_quality(rp) for rp in retrievals]
    qdf = pd.DataFrame(quality)
    qdf.insert(0, 'query', [q[:38] for q in queries[:len(qdf)]])
    display(ui.style_scores(qdf))
    # Publish the reference-free signals to CloudWatch (one record per retrieval).
    for stats in quality:
        obs.emit_retrieval_metrics(stats, kb_id=kb_id, region_name=region)
    mean_util = sum(d['retrieval_utilization'] for d in quality) / len(quality)
    mean_grnd = sum(d['grounded_coverage'] for d in quality) / len(quality)
    ui.ok(f'{len(quality)} agentic retrievals | mean utilization {mean_util:.0%} '
          f'| mean grounded {mean_grnd:.0%}')
    ui.ok(f'Published to {obs.RETRIEVAL_QUALITY_NAMESPACE}')

## Layer 4 — Gateway / MCP metrics

> **📊 What to expect** — the Gateway's view of MCP traffic: `Invocations` (Sum) counts **all** MCP
> ops, not just retrievals (Initialize + ListTools + NotificationsInitialized + InvokeTool per query,
> so **~5× the query count**) · `SystemErrors`/`Throttles` **0** · `UserErrors` may be small/non-zero
> (benign — e.g. the agent's first tool-call format retry) · **`Latency` (Average, ms)** — unlike the
> KB namespace, the Gateway *does* report its own request latency here.

The AgentCore Gateway publishes these to `AWS/Bedrock-AgentCore`. Unlike the KB namespace, this
one **does** include `Latency` (the Gateway measures its own request duration). 

In [ ]:
ui.rule('Layer 4 — Gateway / MCP metrics')
display(ui.style_metrics(obs.get_gateway_metrics(region_name=region, hours=1)))

## Layer 5 — Traces (the agent span tree)

> **📊 What to expect** — a table of OTEL spans (newest hour, ~50–200 rows; **filter to this run's
> `session` prefix**). Per query you'll see the reason–act tree: `AgentCore.Runtime.Invoke` →
> `invoke_agent` → `execute_event_loop_cycle` → `chat` (+ `chat <model>` CLIENT, which carries the
> tokens) → `execute_tool …___AgenticRetrieveStream`. This is the raw material Layers 6 & 7 read.
> The next markdown cell explains why the per-layer *counts* differ.

The deployed agent's OTEL spans, filtered by `session.id`. The tree per query:
`invoke_agent → execute_event_loop_cycle → chat` (the LLM calls) `→ execute_tool kb-retrieve →`
Gateway SERVER/CLIENT. This is the raw material for Layers 6 (tokens on `chat` spans) and 7. 

In [ ]:
ui.rule('Layer 5 — Agent span tree')
spans = obs.query_spans(region_name=region, hours=1, limit=200)
tree = [{'name': s.get('name','?'), 'kind': s.get('kind',''),
         'duration_ms': round(s.get('durationNano',0)/1_000_000, 1),
         'session': s.get('attributes',{}).get('session.id','')[:20]} for s in spans]
print(f'{len(spans)} spans')
display(pd.DataFrame(tree))

### Reading the counts

A natural question: if we sent **3 queries**, why does Layer 4 say `Invocations = 15`, the span
tree shows **6** retrievals, and Layer 6 counts **9** LLM calls? Because each layer counts a
*different thing* — and the gaps are the agentic loop made visible:

| Count | What it measures | Per query |
|---|---|---|
| **3** | agent invocations (`invoke_agent` / `Runtime.Invoke`) | 1 — one per query we sent |
| **6** | KB **retrievals** (`execute_tool …___AgenticRetrieveStream`) | ~2 — the agent calls the tool, reads the result, then calls it again to refine |
| **9** | **LLM calls** (`chat`, Layer 6) | ~3 — reason → (retrieve) → reason → (retrieve) → compose the final answer |
| **15** | Gateway **MCP operations** (Layer 4 `Invocations`) | ~5 — every MCP op: `Initialize` + `ListTools` + `NotificationsInitialized` + `InvokeTool`×2 |

So a single user query is **not** one KB call. A Strands agent runs a reason–act loop: it thinks
(an LLM `chat`), decides to retrieve (a tool call), reads what came back, and often retrieves again
before composing its grounded answer. That's why **LLM calls ≥ retrievals ≥ agent invocations**,
and why the Gateway's MCP-operation count is higher still (it counts protocol handshakes, not just
retrievals). Each number is correct for what it measures — the point of seeing them side by side is
to *understand the shape of the agentic workload*, not to expect them to be equal.

> Only Layer 6's `chat` spans carry token usage, so cost is computed from those (9), not from the
> Gateway's 15 or the 6 retrievals.

## Layer 6 — Cost & quota ⭐

> **📊 What to expect** — three views, all from the agent's `chat` token spans (this session only):
> a **per-call** table (~3 calls/query) · a **session total** (e.g. ~$0.03, ~34k quota tokens) · a
> **per-query rollup** — *cost to serve one KB-grounded answer* (~$0.01/query here). **Cost ≠ quota:**
> cost is flat billing, quota applies the burndown multiplier. ⚠️ Excludes the KB's server-side
> orchestration model (invisible client-side — see the note below).

Each leaf LLM span (`chat <model>`, `kind=CLIENT`) carries `gen_ai.usage.input_tokens` /
`output_tokens`. We compute **two distinct numbers** per call:

- **Cost ($):** `input×price + output×price` — flat, what you're billed.
- **Quota consumed:** `input + output×burndown` — TPM/TPD capacity, *not* dollars.

We count **only the leaf `chat` CLIENT spans** — the INTERNAL `chat` span duplicates the same
counts and `invoke_agent` is a rollup that already includes them (summing all three triple-counts).

> **What this cost does *not* include — the KB's agentic-orchestration model.** Our Gateway target
> uses `foundationModelType="MANAGED"`, so the model that plans + reranks + synthesizes the cited
> answer (inside `AgenticRetrieveStream`) runs **server-side within Bedrock**. Its tokens never reach
> the agent's OTEL telemetry — the `AgenticRetrieveStream` payload records carry no `gen_ai.usage`.
> So Layer 6 measures the **agent's own** model spend (what you can observe client-side); the managed
> orchestration cost is real but invisible here. (Switching the target to a `CUSTOM` model ARN is the
> only way to surface those tokens — out of scope for this notebook.)

In [ ]:
ui.rule('Layer 6 — cost ($, flat) vs quota (burndown)')
def _first(a, *keys):
    for k in keys:
        if a.get(k) not in (None, ''): return a[k]
    return None
priced = []
for s in spans:
    a = s.get('attributes', {}); name = s.get('name',''); kind = s.get('kind','')
    # Only THIS session's spans — query_spans(hours=1) also returns spans from earlier
    # runs in the same hour, which would inflate the cost/quota totals.
    if a.get('session.id') != session_id:
        continue
    if kind == 'CLIENT' and name.startswith('chat'):
        it = _first(a, 'gen_ai.usage.input_tokens', 'gen_ai.usage.prompt_tokens')
        ot = _first(a, 'gen_ai.usage.output_tokens', 'gen_ai.usage.completion_tokens')
        m = _first(a, 'gen_ai.request.model') or model_id
        if it is not None and ot is not None:
            # traceId groups all the spans of ONE user query → lets us roll cost up per query.
            priced.append({'traceId': s.get('traceId'), 'ts': s.get('startTimeUnixNano', 0),
                           'model': m.split('.')[-1], 'in_tok': int(it), 'out_tok': int(ot),
                           'cost_usd': compute_cost_usd(m, int(it), int(ot)),
                           'quota_tokens': quota_consumed(m, int(it), int(ot))})
session_cost, session_quota = 0.0, 0
if priced:
    display(pd.DataFrame([{k: v for k, v in p.items() if k not in ('traceId', 'ts')} for p in priced]))
    session_cost = sum(p['cost_usd'] or 0 for p in priced)
    session_quota = sum(p['quota_tokens'] or 0 for p in priced)
    ui.ok(f'{len(priced)} LLM calls | cost: ${session_cost:.6f}  |  quota consumed: {session_quota:,} tokens')
    ui.info('Cost is flat (billed on actual tokens); quota applies the per-model burndown.')

    # ── Per-query rollup (cost to serve one KB-grounded answer) ──
    # In this notebook the agent exists ONLY to answer from the KB, so every visible token
    # is spent serving a KB query. Grouping by traceId gives cost-per-user-question — the
    # number a team running a KB-backed assistant actually budgets on. (The KB's own
    # server-side orchestration model cost is separate and not visible here — see above.)
    order = []
    for p in sorted(priced, key=lambda x: x['ts']):
        if p['traceId'] not in order:
            order.append(p['traceId'])
    per_query = []
    for i, tid in enumerate(order):
        calls = [p for p in priced if p['traceId'] == tid]
        per_query.append({
            'query': queries[i][:38] if i < len(queries) else (tid or '')[:12],
            'llm_calls': len(calls),
            'in_tok': sum(c['in_tok'] for c in calls),
            'out_tok': sum(c['out_tok'] for c in calls),
            'cost_usd': round(sum(c['cost_usd'] for c in calls), 6),
            'quota_tokens': sum(c['quota_tokens'] for c in calls),
        })
    ui.ok('Cost per KB-grounded answer (grouped by traceId):')
    display(pd.DataFrame(per_query))

    # Publish BOTH the session totals and the per-query series to CloudWatch so the
    # dashboard can graph a session number AND a cost-per-query widget (one line per query).
    obs.emit_cost_metrics(session_cost, session_quota, kb_id=kb_id, region_name=region)
    obs.emit_query_cost_metrics(per_query, kb_id=kb_id, region_name=region)
    ui.ok(f'Published session + per-query cost/quota to {obs.COST_NAMESPACE}')
else:
    ui.info('No leaf LLM spans with tokens yet — spans may still be landing; re-run.')

## Layer 7 — Quality (AgentCore Evaluate) ⭐

> **📊 What to expect** — LLM-as-judge scores, one row per (evaluator × query): `Correctness`,
> `Faithfulness`, `ToolSelectionAccuracy`, each a **`value` 0–1** + a `label` + an `explanation`.
> On this clean single-doc run expect **near-1.0** across the board (mean ~1.00). Scores are **async**
> — if the download cell returns 0 records, wait 1–2 min after traffic and re-run it. Each score also
> reports the judge's own `tokenUsage` (the eval itself costs tokens).

AgentCore **Evaluate** is LLM-as-a-judge over the session's spans. We download the raw span JSON
for this `session_id` from both log groups (`aws/spans` + the Runtime log group) and score it
with built-in evaluators. Each result has a `value` (0–1), `label`, `explanation`, and the
judge's own `tokenUsage`. 

In [ ]:
# Download RAW span JSON (unparsed) for this session, from both log groups.
def _query(log_group, q):
    start = int((time.time() - 3600)); end = int(time.time())
    try:
        qid = logs.start_query(logGroupName=log_group, startTime=start, endTime=end, queryString=q)['queryId']
    except logs.exceptions.ResourceNotFoundException:
        return []
    while (res := logs.get_query_results(queryId=qid))['status'] not in ('Complete', 'Failed'):
        time.sleep(1)
    return res.get('results', []) if res['status'] == 'Complete' else []

def _raw_msgs(log_group):
    q = (f'fields @timestamp, @message | filter ispresent(scope.name) and ispresent(attributes.session.id) '
         f'| filter attributes.session.id = "{session_id}" | sort @timestamp asc')
    rows = _query(log_group, q)
    out = []
    for row in rows:
        for f in row:
            if f['field'] == '@message' and f['value'].strip().startswith('{'):
                try: out.append(json.loads(f['value']))
                except Exception: pass
    return out

runtime_lg = f'/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT'
session_span_logs = _raw_msgs('aws/spans') + _raw_msgs(runtime_lg)
ui.ok(f'Downloaded {len(session_span_logs)} raw span records for the session')

In [ ]:
ui.rule('Layer 7 — quality scores (AgentCore Evaluate)')
evaluators = ['Builtin.Correctness', 'Builtin.Faithfulness', 'Builtin.ToolSelectionAccuracy']
scores = []
if not session_span_logs:
    ui.info('No spans downloaded — wait 1–2 min after traffic and re-run the download cell.')
for ev in evaluators:
    try:
        resp = agentcore_client.evaluate(evaluatorId=ev, evaluationInput={'sessionSpans': session_span_logs})
        for r in resp.get('evaluationResults', []):
            scores.append({'evaluator': ev.split('.')[-1], 'score': r.get('value'),
                           'label': r.get('label',''), 'explanation': (r.get('explanation','') or '')[:80],
                           'error': r.get('errorCode','')})
    except Exception as e:
        scores.append({'evaluator': ev.split('.')[-1], 'score': None, 'label': '', 'explanation': '', 'error': str(e)[:80]})
display(pd.DataFrame(scores))
ok = [s for s in scores if s['score'] is not None]
if ok:
    ui.ok(f'{len(ok)} scores | mean: {sum(s["score"] for s in ok)/len(ok):.2f}')
    # Publish to CloudWatch so the dashboard (below) can graph Layer 7.
    obs.emit_eval_scores(scores, kb_id=kb_id, region_name=region)
    ui.ok(f'Published eval scores to {obs.EVAL_NAMESPACE}')

## The 7-layer CloudWatch dashboard

One `put_dashboard` call unifies all seven layers into a single board. Layers 1, 3, 4 are
CloudWatch metrics; Layer 5 is a span log widget; **Layers 6 (cost/quota, incl. per-query) and 7
(eval scores)** appear because we published them as custom metrics (`BMKB/Cost`, `BMKB/Evaluation`)
in the cells above. `include_eval_cost=True` adds those widgets to the base board from notebook 02.

> **Two timing caveats — both resolved by waiting + re-running:**
> - *Widget shows "No data":* metric **values** take ~2–5 min to populate. Open the dashboard, set a
>   **3h** range, and refresh.
> - *A whole widget is missing* (e.g. the per-query cost or eval-scores widget): the board is built by
>   **discovering** the emitted series via `list_metrics`, which lags emission by 1–5 min. If a widget
>   is absent right after a fast top-to-bottom run, **wait ~2 min and re-run this cell** — the series
>   will have indexed by then and the widget will appear.

In [ ]:
dashboard_name = f'BMKB-Agentic-Eval-{suffix}'
obs.build_kb_dashboard(dashboard_name, kb_id=kb_id, gateway_id=gateway_id,
                       region_name=region, include_eval_cost=True)
ui.rule('7-layer CloudWatch dashboard')
ui.ok(f'Dashboard created: {dashboard_name}')
ui.info(f'https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards/dashboard/{dashboard_name}')

## Summary — every layer, where it lives, how it's captured

| Layer | Signal | Source | Helper |
|-------|--------|--------|--------|
| 1 | KB metrics | `AWS/Bedrock/KnowledgeBases` | `obs.get_kb_metrics()` |
| 2 | Ingestion | ingestion-job API + `APPLICATION_LOGS` | `obs.get_ingestion_stats()` + `obs.query_ingestion_logs()` |
| 3 | Agentic retrieval quality | agent's `AgenticRetrieveStream` payload (runtime log) → `BMKB/RetrievalQuality` | `obs.fetch_agentic_retrievals()` + `obs.extract_agentic_retrieval_quality()` + `obs.emit_retrieval_metrics()` |
| 4 | Gateway metrics | `AWS/Bedrock-AgentCore` | `obs.get_gateway_metrics()` |
| 5 | Span tree | `aws/spans` | `obs.query_spans()` |
| 6 | Cost & quota ⭐ | `gen_ai.usage.*` on spans → `BMKB/Cost` | `pricing.compute_cost_usd()` + `quota_consumed()` + `obs.emit_cost_metrics()` |
| 7 | Quality ⭐ | AgentCore Evaluate → `BMKB/Evaluation` | `bedrock-agentcore.evaluate()` + `obs.emit_eval_scores()` |
| — | **Unified view** | one CloudWatch dashboard | `obs.build_kb_dashboard(include_eval_cost=True)` |

**The two things a deployed agent unlocks:** measured cost (Layer 6, from span tokens) and
quality scores (Layer 7, from Evaluate) — neither available from a local agent. Note also that the
**KB's agentic-orchestration cost is invisible client-side** (see Layer 6): with a service-managed
orchestration model, Bedrock plans and synthesizes server-side, so those tokens never reach the
agent's telemetry. All seven layers land on a single dashboard.

## Cleanup

Tears down **everything created** — Runtime + ECR + toolkit roles, Gateway + role, KB + IAM + S3,
delivery chains, stored ids. Commented; run deliberately. 

In [ ]:
# ui.rule('Cleanup')
# # 0. Dashboard
# try: boto3.client('cloudwatch', region_name=region).delete_dashboards(DashboardNames=[dashboard_name])
# except Exception as e: ui.info(f'dashboard: {e}')
# # 1. Runtime + ECR + toolkit-created roles.
# #    delete_agent_runtime needs the FULL runtime id (name-suffix), not just the suffix.
# try: ac.delete_agent_runtime(agentRuntimeId=agent_id)
# except Exception as e: ui.info(f'runtime: {e}')
# try:
#     boto3.client('ecr', region_name=region).delete_repository(
#         repositoryName=f'bedrock-agentcore-{agent_name}', force=True)
# except Exception as e: ui.info(f'ecr: {e}')
# try: iam.delete_role_policy(RoleName=exec_role_name, PolicyName='invoke-gateway')
# except Exception: pass
# # Toolkit-created exec + codebuild roles (random hex suffix) — delete both.
# for _r in [x['RoleName'] for x in iam.list_roles(MaxItems=500)['Roles']
#            if x['RoleName'].startswith(('AmazonBedrockAgentCoreSDKRuntime-', 'AmazonBedrockAgentCoreSDKCodeBuild-'))]:
#     try:
#         for _p in iam.list_role_policies(RoleName=_r).get('PolicyNames', []): iam.delete_role_policy(RoleName=_r, PolicyName=_p)
#         for _ap in iam.list_attached_role_policies(RoleName=_r).get('AttachedPolicies', []): iam.detach_role_policy(RoleName=_r, PolicyArn=_ap['PolicyArn'])
#         iam.delete_role(RoleName=_r)
#     except Exception as e: ui.info(f'toolkit role {_r}: {e}')
# # 2. Gateway + delivery + role
# for t in ac.list_gateway_targets(gatewayIdentifier=gateway_id).get('items', []):
#     ac.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=t['targetId'])
# time.sleep(5); ac.delete_gateway(gatewayIdentifier=gateway_id)
# for d in logs.describe_deliveries().get('deliveries', []):
#     if d.get('deliverySourceName') == f'gw-{gateway_id}-traces': logs.delete_delivery(id=d['id'])
# for nm, fn in [(f'gw-{gateway_id}-traces', logs.delete_delivery_source), (f'gw-{gateway_id}-xray', logs.delete_delivery_destination)]:
#     try: fn(name=nm)
#     except Exception: pass
# try: iam.delete_role_policy(RoleName=gw_role_name, PolicyName='kb-retrieve'); iam.delete_role(RoleName=gw_role_name)
# except Exception as e: ui.info(f'gw role: {e}')
# # 3. KB + IAM + S3
# if hasattr(kb, '_data_sources'):
#     kb.delete_kb(delete_iam=True, delete_s3_bucket=True)
# else:
#     ManagedKnowledgeBase.delete_kb_by_id(kb_id, region_name=region)
#     kb_role = f'AmazonBedrockExecutionRoleForKnowledgeBase_{suffix}'
#     for pol in (f'AmazonBedrockCloudWatchPolicyForKnowledgeBase_{suffix}', f'AmazonBedrockS3PolicyForKnowledgeBase_{suffix}'):
#         arn = f'arn:aws:iam::{account_id}:policy/{pol}'
#         try: iam.detach_role_policy(RoleName=kb_role, PolicyArn=arn)
#         except Exception: pass
#         try: iam.delete_policy(PolicyArn=arn)
#         except Exception: pass
#     try: iam.delete_role(RoleName=kb_role)
#     except Exception as e: ui.info(f'kb role: {e}')
#     try:
#         b = boto3.resource('s3').Bucket(bucket_name); b.objects.all().delete(); b.delete()
#     except Exception as e: ui.info(f'bucket: {e}')
# %store -d kb_id
# %store -d gateway_id
# ui.ok('Cleanup complete.')